In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Library

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Flatten, Dense, Embedding, Dropout, Concatenate, MultiHeadAttention, Lambda
from tensorflow.keras.optimizers import Adam


from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
def data_split(df, test_size=0.2, valid_size=0.125, random_state=42):

    # 1차 분할: train+valid, test
    df_train, df_test = train_test_split(df, test_size=test_size, random_state=random_state)

    # 2차 분할: train, valid
    df_train, df_valid = train_test_split(df_train, test_size=valid_size, random_state=random_state)

    df_train = df_train.reset_index(drop=True)
    df_valid = df_valid.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    'Training Dataset'
    user_id_train = df_train['user_id'].values
    item_id_train = df_train['asin'].values
    roberta_train = np.array(df_train['roberta_emb'].tolist())
    vgg16_train = np.array(df_train['VGG16_emb'].tolist())
    y_train = df_train['rating'].values

    'Valid Dataset'
    user_id_valid = df_valid['user_id'].values
    item_id_valid = df_valid['asin'].values
    roberta_valid = np.array(df_valid['roberta_emb'].tolist())
    vgg16_valid = np.array(df_valid['VGG16_emb'].tolist())
    y_valid = df_valid['rating'].values

    'Test Dataset'
    user_id_test = df_test['user_id'].values
    item_id_test = df_test['asin'].values
    roberta_test = np.array(df_test['roberta_emb'].tolist())
    vgg16_test = np.array(df_test['VGG16_emb'].tolist())
    y_test = df_test['rating'].values

    return (
        user_id_train, item_id_train, roberta_train, vgg16_train, y_train,
        user_id_valid, item_id_valid, roberta_valid, vgg16_valid, y_valid,
        user_id_test, item_id_test, roberta_test, vgg16_test, y_test
    )

# Load Data

In [ ]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_baseline_emb.pkl')

print(df.shape)
df.head()

In [ ]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
print(n_users)
n_items = df['asin'].nunique()
print(n_items)

In [ ]:
user_id_train, item_id_train, roberta_train, vgg16_train, y_train, \
user_id_valid, item_id_valid, roberta_valid, vgg16_valid, y_valid, \
user_id_test, item_id_test, roberta_test, vgg16_test, y_test = data_split(df)

# Model

In [ ]:
def CAMRec(num_users, num_items, dropout, learning_rate):

    # 사용자 ID
    user_id_input = Input(shape=(1,), name='user_id_input')
    user_emb = Embedding(input_dim=num_users, output_dim=128, name='user_embedding')(user_id_input)
    user_emb = Flatten()(user_emb)

    # 아이템 ID
    item_id_input = Input(shape=(1,), name='item_id_input')
    item_emb = Embedding(input_dim=num_items, output_dim=128, name='item_embedding')(item_id_input)
    item_emb = Flatten()(item_emb)

    # Text Input
    roberta_input = Input(shape=(768,), name='roberta_input')

    # Image Input
    vgg16_input = Input(shape=(4096,), name='vgg16_input')

    # Interaction MLP
    ncf_nlp = Concatenate()([user_emb, item_emb])
    ncf_nlp = Dense(128, activation='relu')(ncf_nlp)
    ncf_nlp = Dropout(dropout)(ncf_nlp)
    ncf_nlp = Dense(64, activation='relu')(ncf_nlp)
    ncf_nlp = Dropout(dropout)(ncf_nlp)

    # Text MLP
    roberta_nlp = Dense(512, activation='relu')(roberta_input)
    roberta_nlp = Dropout(dropout)(roberta_nlp)
    roberta_nlp = Dense(256, activation='relu')(roberta_nlp)
    roberta_nlp = Dropout(dropout)(roberta_nlp)

    # Image MLP
    vgg16_nlp = Dense(512, activation='relu')(vgg16_input)
    vgg16_nlp = Dropout(dropout)(vgg16_nlp)
    vgg16_nlp = Dense(256, activation='relu')(vgg16_nlp)
    vgg16_nlp = Dropout(dropout)(vgg16_nlp)

    # Co-Attention Input
    roberta_vec = Lambda(lambda x: tf.expand_dims(x, axis=1), name='roberta_expand')(roberta_nlp)
    vgg16_vec = Lambda(lambda x: tf.expand_dims(x, axis=1), name='vgg16_expand')(vgg16_nlp)

    # Co-Attention(bi-directional cross attention)
    text_attention = MultiHeadAttention(num_heads=4, key_dim=64, name='roberta_cross_attention')(query=roberta_vec, value=vgg16_vec, key=vgg16_vec)
    img_attention = MultiHeadAttention(num_heads=4, key_dim=64, name='vgg16_cross_attention')(query=vgg16_vec, value=roberta_vec, key=roberta_vec)

    text_attention = Lambda(lambda x: tf.squeeze(x, axis=1), name='roberta_squeeze')(text_attention) # [batch, emb]
    img_attention = Lambda(lambda x: tf.squeeze(x, axis=1), name='vgg16_squeeze')(img_attention) # [batch, emb]

    multi_modal_rep = Concatenate(axis=-1)([text_attention, img_attention]) # [batch, emb*2]

    # Co-Attention MLP
    multi_modal_rep = Dense(256, activation='relu')(multi_modal_rep)
    multi_modal_rep = Dropout(dropout)(multi_modal_rep)

    # Final
    prediction = Concatenate(name='all_representation')([ncf_nlp, multi_modal_rep])
    prediction = Dense(64, activation='relu')(prediction)
    final = Dense(1, activation='linear', name='rating_prediction')(prediction)

    model = Model(inputs=[user_id_input, item_id_input, roberta_input, vgg16_input],
                  outputs=final)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mean_squared_error', metrics=['mean_absolute_error', 'mean_squared_error'])

    return model

In [ ]:
dropout = 0.0
learning_rate = 0.001

In [ ]:
model = CAMRec(n_users, n_items, dropout, learning_rate)
model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor ='val_loss', min_delta = 0.001, patience = 5, verbose = 1, mode = 'min', restore_best_weights = True)

In [ ]:
history = model.fit([user_id_train, item_id_train, roberta_train, vgg16_train],
                    y_train,
                    validation_data = ([user_id_valid, item_id_valid, roberta_valid, vgg16_valid], y_valid),
                    epochs = 100,
                    batch_size = 256,
                    callbacks=[early_stopping])

In [ ]:
# 모델 성능 측정
predicted_ratings = model.predict([user_id_test, item_id_test, roberta_test, vgg16_test])

MAE = mean_absolute_error(y_test, predicted_ratings)
MSE = mean_squared_error(y_test, predicted_ratings)
RMSE = np.sqrt(MSE)
MAPE = mean_absolute_percentage_error(y_test, predicted_ratings) * 100


print(f'MAE: {MAE:.4f}')
print(f'MSE: {MSE:.4f}')
print(f'RMSE: {RMSE:.4f}')
print(f'MAPE: {MAPE:.4f}%')